In [309]:
import json
import re
from typing import *

import huggingface_hub
import regex
import tabulate
import tokenizers
import tokenizers.pre_tokenizers
import transformers

In [98]:
tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
path = huggingface_hub.hf_hub_download("meta-llama/Llama-3.2-1B-Instruct", "tokenizer.json")
print(path)
with open(path, "r") as f:
    tokenizer_data = json.load(f)

/home/douglaso/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/e9f8effbab1cbdc515c11ee6e098e3d5a9f51e14/tokenizer.json


## Example

In [353]:
test_text = "This ISN'T my number:     1234567890.\n\nSuperfluously λ \x7f!!!!!"

print("         Text:", repr(test_text))
print("Pre-tokenized:", [s for s, _ in tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(test_text)])
ref_tokens = tokenizer.encode(test_text)
print("    Tokenized:", tokenizer.convert_ids_to_tokens(ref_tokens))
print("          IDs:", ref_tokens)
ref_decoded = tokenizer.decode(ref_tokens[1:])
print("      Decoded:", repr(ref_decoded))
assert ref_decoded == test_text

         Text: "This ISN'T my number:     1234567890.\n\nSuperfluously λ \x7f!!!!!"
Pre-tokenized: ['This', 'ĠISN', "'T", 'Ġmy', 'Ġnumber', ':', 'ĠĠĠĠ', 'Ġ', '123', '456', '789', '0', '.ĊĊ', 'Superfluously', 'ĠÎ»', 'Ġġ!!!!!']
    Tokenized: ['<|begin_of_text|>', 'This', 'ĠIS', 'N', "'T", 'Ġmy', 'Ġnumber', ':', 'ĠĠĠĠ', 'Ġ', '123', '456', '789', '0', '.ĊĊ', 'Super', 'flu', 'ously', 'ĠÎ»', 'Ġ', 'ġ', '!!!!!']
          IDs: [128000, 2028, 3507, 45, 17773, 856, 1396, 25, 257, 220, 4513, 10961, 16474, 15, 382, 19841, 27256, 7162, 49438, 220, 221, 70900]
      Decoded: "This ISN'T my number:     1234567890.\n\nSuperfluously λ \x7f!!!!!"


## Components

In [193]:
print("Added tokens:")
print(len(tokenizer_data["added_tokens"]))
display(tokenizer_data["added_tokens"][0])
print("Pre-tokenizer:")
display(tokenizer_data["pre_tokenizer"])
print("Post-processor:")
display(tokenizer_data["post_processor"])
print("Decoder:")
display(tokenizer_data["decoder"])
print("Model:")
display({k: f"{type(v)}{{x{len(v)}}}" if isinstance(v, (list, dict)) else str(v) for k, v in tokenizer_data["model"].items()})

Added tokens:
256


{'id': 128000,
 'content': '<|begin_of_text|>',
 'single_word': False,
 'lstrip': False,
 'rstrip': False,
 'normalized': False,
 'special': True}

Pre-tokenizer:


{'type': 'Sequence',
 'pretokenizers': [{'type': 'Split',
   'pattern': {'Regex': "(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"},
   'behavior': 'Isolated',
   'invert': False},
  {'type': 'ByteLevel',
   'add_prefix_space': False,
   'trim_offsets': True,
   'use_regex': False}]}

Post-processor:


{'type': 'Sequence',
 'processors': [{'type': 'ByteLevel',
   'add_prefix_space': True,
   'trim_offsets': False,
   'use_regex': True},
  {'type': 'TemplateProcessing',
   'single': [{'SpecialToken': {'id': '<|begin_of_text|>', 'type_id': 0}},
    {'Sequence': {'id': 'A', 'type_id': 0}}],
   'pair': [{'SpecialToken': {'id': '<|begin_of_text|>', 'type_id': 0}},
    {'Sequence': {'id': 'A', 'type_id': 0}},
    {'SpecialToken': {'id': '<|begin_of_text|>', 'type_id': 1}},
    {'Sequence': {'id': 'B', 'type_id': 1}}],
   'special_tokens': {'<|begin_of_text|>': {'id': '<|begin_of_text|>',
     'ids': [128000],
     'tokens': ['<|begin_of_text|>']}}}]}

Decoder:


{'type': 'ByteLevel',
 'add_prefix_space': True,
 'trim_offsets': True,
 'use_regex': True}

Model:


{'type': 'BPE',
 'dropout': 'None',
 'unk_token': 'None',
 'continuing_subword_prefix': 'None',
 'end_of_word_suffix': 'None',
 'fuse_unk': 'False',
 'byte_fallback': 'False',
 'ignore_merges': 'True',
 'vocab': "<class 'dict'>{x128000}",
 'merges': "<class 'list'>{x280147}"}

## 1. Split pretokenizer

In [396]:
pattern = regex.compile(tokenizer_data["pre_tokenizer"]["pretokenizers"][0]["pattern"]["Regex"])
print(pattern.pattern)
rep_split = pattern.findall(test_text)
print(rep_split)

(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+
['This', ' ISN', "'T", ' my', ' number', ':', '    ', ' ', '123', '456', '789', '0', '.\n\n', 'Superfluously', ' λ', ' \x7f!!!!!']


## 2. Byte pretokenizer

In [285]:
def byte_to_unicode(i: int) -> str:
    if i < 0x21:
        return chr(0x100 + i)
    if 0x7f <= i and i < 0xa1:
        return chr(0x100 + (-0x7f + 0x21) + i)
    if i == 0xad:
        return chr(0x100 + (-0xad + 0x21 + (0xa1 - 0x7f)) + i)
    return chr(i)

byte_map = [byte_to_unicode(i) for i in range(256)]
rep_pre = ["".join(byte_map[b] for b in token.encode("utf8")) for token in rep_split]
print(rep_pre)

_unk = [(f"{n:02x}", i) for n, i in enumerate(byte_map) if i not in tokenizer_data["model"]["vocab"]]
assert not _unk, _unk

['This', 'ĠISN', "'T", 'Ġmy', 'Ġnumber', ':', 'ĠĠĠĠ', 'Ġ', '123', '456', '789', '0', '.ĊĊ', 'Superfluously', 'ĠÎ»', 'Ġġ!!!!!']


## 3. BPE

In [336]:
def tokenize_bpe(merges: List[str], pre_tokens: List[str]) -> List[str]:
    words = [list(t) for t in pre_tokens]
    for merge in merges:
        a, b = merge.split(" ")
        for word_idx, word_tokens in enumerate(words):
            idx = 0
            while idx < len(word_tokens) - 1:
                if word_tokens[idx:idx+2] == [a, b]:
                    del word_tokens[idx + 1]
                    word_tokens[idx] = a + b
                else:
                    idx += 1
            words[word_idx] = word_tokens
    return [token for word in words for token in word]

rep_bpe = tokenize_bpe(tokenizer_data["model"]["merges"], rep_pre)
print(rep_bpe)
bos_id, = (t["id"] for t in tokenizer_data["added_tokens"] if t["content"] == "<|begin_of_text|>")
rep_tokens = [bos_id] + [tokenizer_data["model"]["vocab"][t] for t in rep_bpe]
print(rep_tokens)
assert rep_tokens == ref_tokens

['This', 'ĠIS', 'N', "'T", 'Ġmy', 'Ġnumber', ':', 'ĠĠĠĠ', 'Ġ', '123', '456', '789', '0', '.ĊĊ', 'Super', 'flu', 'ously', 'ĠÎ»', 'Ġ', 'ġ', '!!!!!']
[128000, 2028, 3507, 45, 17773, 856, 1396, 25, 257, 220, 4513, 10961, 16474, 15, 382, 19841, 27256, 7162, 49438, 220, 221, 70900]


## Reconstruction

In [381]:
id_to_token = {id: token for token, id in tokenizer_data["model"]["vocab"].items()}
assert all([len(b)==1 for b in byte_map])
byte_unmap = {c: bytes([i]) for i, c in enumerate(byte_map)}

rep_decoded = (b"".join(b"".join(byte_unmap[c] for c in id_to_token[t]) for t in rep_tokens if t != bos_id)).decode()
print(repr(rep_decoded))
assert rep_decoded == ref_decoded

"This ISN'T my number:     1234567890.\n\nSuperfluously λ \x7f!!!!!"
